# ThermoRG Phase B — Session 1: Full Training Validation

## Goal
Train 6 ThermoNet architectures to convergence (50 epochs) on CIFAR-10,
measure final test accuracy, and verify J_topo → E_floor correlation.

**Baseline:** ResNet-18 achieves ~94.8% on CIFAR-10 (literature value, not retrained).

**Success criterion:** ThermoNet with good J_topo achieves comparable accuracy
to ResNet-18 while using significantly fewer parameters.

## Candidates

| ID | J_topo | Params | Selection Reason |
|----|--------|--------|-------------------|
| T18 | 0.789 | 0.59M | Best J_topo |
| T04 | 0.763 | 0.93M | No-skip control |
| T11 | 0.505 | 1.05M | Threshold boundary |
| T09 | 0.749 | 1.31M | High J_topo, mid params |
| T13 | 0.678 | 1.08M | Good J_topo, small params |
| T16 | 0.524 | 2.05M | Threshold boundary |

**Reference baseline:** ResNet-18 (11.7M) → ~94.8% (literature)

In [ ]:
import os, sys, time, json, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print(f"Start time: {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
# ============================================================
# ThermoNet Model Builders
# ============================================================

def scale_channels(chs, mult):
    return [chs[0]] + [max(1, int(c * mult)) for c in chs[1:]]

class ConvBlock(nn.Module):
    def __init__(self, ic, oc, act='gelu', norm=True):
        super().__init__()
        self.conv = nn.Conv2d(ic, oc, 3, padding=1, bias=not norm)
        self.norm = nn.LayerNorm([oc, 32, 32]) if norm else nn.Identity()
        if act == 'gelu':
            self.act = nn.GELU()
        elif act == 'tga':
            self.act = nn.Tanh()
        else:
            self.act = nn.ReLU(inplace=True)
    
    def forward(self, x):
        return self.act(self.norm(self.conv(x)))

def build_TN3(wm=1.0, num_classes=10, use_skip=True):
    ch = scale_channels([3, 64, 64, 128, 128], wm)
    blocks = nn.ModuleList()
    for i in range(len(ch) - 1):
        blocks.append(ConvBlock(ch[i], ch[i+1], 'gelu', True))
    pool = nn.AdaptiveAvgPool2d((1, 1))
    fc = nn.Linear(ch[-1], num_classes)
    return nn.Sequential(*[*blocks, pool, nn.Flatten(), fc])

def build_TN5(wm=1.0, num_classes=10, use_skip=True):
    ch = scale_channels([3, 64, 128, 256, 128, 64], wm)
    blocks = nn.ModuleList()
    for i in range(len(ch) - 1):
        blocks.append(ConvBlock(ch[i], ch[i+1], 'gelu', True))
    pool = nn.AdaptiveAvgPool2d((1, 1))
    fc = nn.Linear(ch[-1], num_classes)
    return nn.Sequential(*[*blocks, pool, nn.Flatten(), fc])

def build_TN7(wm=1.0, num_classes=10, use_skip=True):
    ch = scale_channels([3, 64, 64, 128, 128, 256, 128, 64], wm)
    blocks = nn.ModuleList()
    for i in range(len(ch) - 1):
        blocks.append(ConvBlock(ch[i], ch[i+1], 'tga', True))
    pool = nn.AdaptiveAvgPool2d((1, 1))
    fc = nn.Linear(ch[-1], num_classes)
    return nn.Sequential(*[*blocks, pool, nn.Flatten(), fc])

def build_TN9(wm=1.0, num_classes=10, use_skip=False):
    ch = scale_channels([3] + [64]*8, wm)
    blocks = nn.ModuleList()
    for i in range(len(ch) - 1):
        blocks.append(ConvBlock(ch[i], ch[i+1], 'gelu', True))
    pool = nn.AdaptiveAvgPool2d((1, 1))
    fc = nn.Linear(ch[-1], num_classes)
    return nn.Sequential(*[*blocks, pool, nn.Flatten(), fc])

def build_TN_arbitrary_depth(depth, wm=1.0, num_classes=10, use_skip=True):
    if depth == 3:
        return build_TN3(wm, num_classes, use_skip)
    elif depth == 5:
        return build_TN5(wm, num_classes, use_skip)
    elif depth == 7:
        return build_TN7(wm, num_classes, use_skip)
    elif depth == 9:
        return build_TN9(wm, num_classes, use_skip)
    else:
        base_pattern = [64, 128, 256, 512]
        ch = [3]
        reps = (depth + 3) // 4
        for _ in range(reps):
            ch.extend(base_pattern)
        ch = ch[:depth+1]
        while len(ch) < depth + 1:
            ch.append(64)
        ch = scale_channels(ch, wm)
        blocks = nn.ModuleList()
        for i in range(len(ch) - 1):
            blocks.append(ConvBlock(ch[i], ch[i+1], 'gelu', True))
        pool = nn.AdaptiveAvgPool2d((1, 1))
        fc = nn.Linear(ch[-1], num_classes)
        return nn.Sequential(*[*blocks, pool, nn.Flatten(), fc])

In [ ]:
# ============================================================
# J_topo Computation (for reference, from initialized model)
# ============================================================

def compute_D_eff_power_iteration(W, n_iter=20):
    W_flat = W.reshape(W.shape[0], -1)
    if min(W_flat.shape) == 0:
        return 1.0
    v = torch.randn(W_flat.shape[1], device=W_flat.device)
    v = v / (v.norm() + 1e-10)
    for _ in range(n_iter):
        Wv = torch.matmul(W_flat.T, torch.matmul(W_flat, v))
        v_new = Wv / (Wv.norm() + 1e-10)
        if torch.abs(v - v_new).sum() < 1e-8:
            break
        v = v_new
    lambda_max_sq = torch.matmul(W_flat.T, torch.matmul(W_flat, v)).norm()**2 / (v.norm()**2 + 1e-10)
    fro_sq = (W_flat ** 2).sum()
    return float((fro_sq / (lambda_max_sq + 1e-10)).clamp(min=1.0))

def compute_J_topo(model):
    import re
    skip_exclude = ['layernorm', 'layer_norm', 'norm', 'batchnorm', 'bn', 
                    'pool', 'flatten', 'fc', 'linear']
    eta_list = []
    prev_D_eff = None
    
    for name, module in model.named_modules():
        if any(re.search(p, name.lower()) for p in skip_exclude):
            eta_list.append(1.0)
            continue
        if not hasattr(module, 'weight') or module.weight is None:
            continue
        W = module.weight.data
        if W.numel() == 0:
            continue
        W_flat = W.reshape(W.shape[0], -1)
        if min(W_flat.shape) == 0:
            continue
        D_eff = compute_D_eff_power_iteration(W, n_iter=20)
        if prev_D_eff is not None:
            eta = D_eff / max(prev_D_eff, 1.0)
            eta_list.append(float(eta))
        prev_D_eff = D_eff
    
    if not eta_list:
        return 1.0
    log_etas = [abs(math.log(max(eta, 1e-10))) for eta in eta_list]
    return math.exp(-np.mean(log_etas))

In [ ]:
# ============================================================
# Data Loading
# ============================================================

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465], [0.2023, 0.1994, 0.2010])
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465], [0.2023, 0.1994, 0.2010])
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")

In [ ]:
# ============================================================
# Training Functions
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += X.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            out = model(X)
            loss = F.cross_entropy(out, y)
            total_loss += loss.item() * X.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += X.size(0)
    return total_loss / total, correct / total

def train_full(model, train_loader, test_loader, epochs=50, lr=0.01, seed=42):
    """Train to convergence and track best model."""
    torch.manual_seed(seed)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    
    best_acc = 0.0
    best_epoch = 0
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
    
    for epoch in range(epochs):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        test_loss, test_acc = evaluate(model, test_loader)
        scheduler.step()
        elapsed = time.time() - t0
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        
        if test_acc > best_acc:
            best_acc = test_acc
            best_epoch = epoch + 1
        
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d}/{epochs}: "
                  f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
                  f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f} ({elapsed:.1f}s)")
    
    return {
        'best_acc': best_acc,
        'best_epoch': best_epoch,
        'final_acc': history['test_acc'][-1],
        'final_loss': history['test_loss'][-1],
        'history': history
    }

In [ ]:
# ============================================================
# 6 Candidates to Validate
# ============================================================

CANDIDATES = [
    {'id': 'T18', 'depth': 9,  'width_mult': 0.5,  'use_skip': True,  'J_topo': 0.789},
    {'id': 'T04', 'depth': 9,  'width_mult': 0.75, 'use_skip': False, 'J_topo': 0.763},
    {'id': 'T11', 'depth': 3,  'width_mult': 1.0,  'use_skip': True,  'J_topo': 0.505},
    {'id': 'T09', 'depth': 9,  'width_mult': 1.0,  'use_skip': True,  'J_topo': 0.749},
    {'id': 'T13', 'depth': 7,  'width_mult': 0.5,  'use_skip': True,  'J_topo': 0.678},
    {'id': 'T16', 'depth': 5,  'width_mult': 1.0,  'use_skip': True,  'J_topo': 0.524},
]

EPOCHS = 50
SEED = 42

# Reference baseline
RESNET18_ACC = 0.948  # Literature value (not retrained)

print(f"Candidates: {[c['id'] for c in CANDIDATES]}")
print(f"Epochs: {EPOCHS}")
print(f"Reference: ResNet-18 accuracy = {RESNET18_ACC:.1%}")
print(f"Started: {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
# ============================================================
# Train All 6 Candidates
# ============================================================

results = []
t_start = time.time()

for i, config in enumerate(CANDIDATES):
    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(CANDIDATES)}] {config['id']}: "
          f"depth={config['depth']}, wm={config['width_mult']}, skip={config['use_skip']}")
    print(f"  Pre-computed J_topo: {config['J_topo']:.4f}")
    
    # Build model
    torch.manual_seed(SEED)
    model = build_TN_arbitrary_depth(
        config['depth'], config['width_mult'], 10, config['use_skip']
    ).to(DEVICE)
    
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters: {n_params/1e6:.2f}M")
    
    # Train
    print(f"  Training {EPOCHS} epochs...")
    t0 = time.time()
    result = train_full(model, train_loader, test_loader, epochs=EPOCHS, seed=SEED)
    train_time = time.time() - t0
    
    results.append({
        'id': config['id'],
        'depth': config['depth'],
        'width_mult': config['width_mult'],
        'use_skip': config['use_skip'],
        'J_topo': config['J_topo'],
        'n_params': n_params,
        'best_acc': result['best_acc'],
        'best_epoch': result['best_epoch'],
        'final_acc': result['final_acc'],
        'final_loss': result['final_loss'],
        'train_time_min': train_time / 60,
        'history': result['history'],
    })
    
    print(f"  → Best: acc={result['best_acc']:.4f} at epoch {result['best_epoch']}")
    print(f"  → Final: acc={result['final_acc']:.4f}, loss={result['final_loss']:.4f}")
    print(f"  → Time: {train_time/60:.1f} min")

t_total = time.time() - t_start
print(f"\n{'='*60}")
print(f"Total training time: {t_total/60:.1f} minutes ({t_total/3600:.2f} hours)")

In [ ]:
# ============================================================
# Results Summary
# ============================================================

from scipy.stats import spearmanr

print("\n" + "="*70)
print("RESULTS SUMMARY")
print("="*70)
print(f"{'ID':<6} {'J_topo':<8} {'Params':<10} {'Best Acc':<12} {'Final Acc':<12} {'Time (min)':<10}")
print("-"*70)

# Sort by J_topo
by_J = sorted(results, key=lambda x: -x['J_topo'])
for r in by_J:
    comparable = '⭐' if r['final_acc'] >= RESNET18_ACC - 0.02 else ''
    print(f"{r['id']:<6} {r['J_topo']:<8.4f} {r['n_params']/1e6:<10.2f}M "
          f"{r['best_acc']:<12.4f} {r['final_acc']:<12.4f} "
          f"{r['train_time_min']:<10.1f} {comparable}")

print(f"\nReference: ResNet-18 → {RESNET18_ACC:.4f} (literature, not retrained)")
print(f"⭐ = within 2% of ResNet-18")

# Sort by accuracy
by_acc = sorted(results, key=lambda x: -x['final_acc'])
print("\nRanked by Test Accuracy (descending):")
for rank, r in enumerate(by_acc, 1):
    gap = r['final_acc'] - RESNET18_ACC
    print(f"  {rank}. {r['id']}: {r['final_acc']:.4f} (gap vs ResNet-18: {gap:+.4f})")

# Compute Spearman correlation
J_vals = [r['J_topo'] for r in results]
acc_vals = [r['final_acc'] for r in results]

r_spearman, p_val = spearmanr(J_vals, acc_vals)
print(f"\nSpearman(J_topo, final_acc): r = {r_spearman:.4f}, p = {p_val:.4f}")

print("\nINTERPRETATION:")
if abs(r_spearman) >= 0.7:
    print(f"  |r| >= 0.7: STRONG — J_topo predicts accuracy")
elif abs(r_spearman) >= 0.5:
    print(f"  |r| >= 0.5: MODERATE — J_topo is useful but not definitive")
else:
    print(f"  |r| < 0.5: WEAK — J_topo alone insufficient for accuracy")

In [ ]:
# ============================================================
# Save Results
# ============================================================

output = {
    'timestamp': datetime.now().isoformat(),
    'epochs': EPOCHS,
    'seed': SEED,
    'reference_resnet18_acc': RESNET18_ACC,
    'candidates': [
        {k: v for k, v in r.items() if k != 'history'} 
        for r in results
    ],
    'spearman': {'r': r_spearman, 'p': p_val},
    'total_time_minutes': t_total / 60,
}

out_path = '/kaggle/working/phase_b_session1_results.json'
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f"Results saved to {out_path}")

In [ ]:
# ============================================================
# Visualization
# ============================================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Training curves (test accuracy)
ax1 = axes[0]
for r in results:
    ax1.plot(r['history']['test_acc'], label=f"{r['id']} (J={r['J_topo']:.3f})")
ax1.axhline(y=RESNET18_ACC, color='black', linestyle='--', label='ResNet-18')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Test Accuracy')
ax1.set_title('Test Accuracy vs Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: J_topo vs Final Test Accuracy
ax2 = axes[1]
for r in results:
    ax2.scatter(r['J_topo'], r['final_acc'], s=100, c='blue')
    ax2.annotate(r['id'], (r['J_topo'], r['final_acc']), 
                 xytext=(5, 5), textcoords='offset points')
ax2.axhline(y=RESNET18_ACC, color='black', linestyle='--', label='ResNet-18')
ax2.set_xlabel('J_topo (higher = better)')
ax2.set_ylabel('Test Accuracy')
ax2.set_title(f'J_topo vs Test Accuracy (Spearman r={r_spearman:.3f})')
ax2.grid(True, alpha=0.3)
ax2.legend()

# Plot 3: Params vs Accuracy (efficiency)
ax3 = axes[2]
for r in results:
    ax3.scatter(r['n_params']/1e6, r['final_acc'], s=100, c='green')
    ax3.annotate(r['id'], (r['n_params']/1e6, r['final_acc']),
                 xytext=(5, 5), textcoords='offset points')
ax3.axhline(y=RESNET18_ACC, color='black', linestyle='--', label='ResNet-18')
ax3.set_xlabel('Parameters (M)')
ax3.set_ylabel('Test Accuracy')
ax3.set_title('Params vs Accuracy (efficiency)')
ax3.grid(True, alpha=0.3)
ax3.legend()

plt.tight_layout()
plt.savefig('/kaggle/working/phase_b_session1.png', dpi=150)
plt.show()

print("Figure saved to /kaggle/working/phase_b_session1.png")